# Notebook 03 — Graphe de Connaissances Neo4j
## Module 03 · Système de Recommandation Hybride Emploi-Compétences · Cameroun
**NGOULOU-NGOUBILI Irch Defluviaire · ISE M2 · Data Science & Marketing**

---

### Objectif
Charger les 16 types de nœuds et 22 types de relations du graphe de connaissances
dans Neo4j à partir des données normalisées (Module 01) et des référentiels ESCO, MEPC, NCF.

### Plan
1. Vérification des pré-requis et connexion Neo4j
2. Création du schéma (contraintes + index)
3. Chargement ESCO (Compétences, Métiers, ISCO)
4. Chargement MEPC (3 niveaux + mapping ISCO)
5. Chargement NCF (4 niveaux)
6. Chargement Offres + nœuds contextuels
7. Chargement Candidats
8. Validation et métriques du graphe
9. Requêtes de démonstration (Skill Gap, matching, roadmap)
10. Visualisation de la structure du graphe

> **Pré-requis** : Neo4j Desktop ou Neo4j Community Edition installé et démarré
> URI : `bolt://localhost:7687` | User : `neo4j` | Password : à configurer dans `config_neo4j.py`

## 1. Vérification des pré-requis et connexion

In [ ]:
import sys, json, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

ROOT    = Path('../..').resolve()
PROC    = ROOT / 'data' / 'processed'
SRC     = ROOT / 'src' / '03_knowledge_graph'
sys.path.insert(0, str(SRC))

# Vérification des données disponibles
print('=== DONNÉES DISPONIBLES ===')
for f in sorted(PROC.iterdir()):
    if f.suffix == '.parquet':
        df = pd.read_parquet(f)
        print(f'  {f.name:<45} {df.shape[0]:>6} lignes x {df.shape[1]} cols')

# Vérification des CSVs ESCO
ESCO_DIR = Path('/mnt/user-data/uploads')
print('\n=== CSVS ESCO ===')
for f in ['skills_fr.csv','occupations_fr.csv','occupationSkillRelations_fr.csv',
          'skillsHierarchy_fr.csv','ISCOGroups_fr.csv']:
    p = ESCO_DIR / f
    if p.exists():
        with open(p) as fh: n = sum(1 for _ in fh)
        print(f'  {f:<45} {n:>8,} lignes')

In [ ]:
# Test de connexion Neo4j
NEO4J_AVAILABLE = False  # Mettre True si Neo4j est démarré

if NEO4J_AVAILABLE:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver('bolt://localhost:7687',
                                  auth=('neo4j','password'))
    driver.verify_connectivity()
    print('Connexion Neo4j OK')
    # Compter les nœuds existants
    with driver.session() as s:
        r = s.run('MATCH (n) RETURN labels(n)[0] AS label, count(n) AS n').data()
        for row in r: print(f'  {row["label"]}: {row["n"]:,}')
else:
    print('Mode démonstration — Neo4j non connecté')
    print('Pour démarrer : neo4j start (ou Neo4j Desktop)')
    print('Config : src/03_knowledge_graph/config_neo4j.py')

## 2. Création du schéma (contraintes + index)

Le schéma garantit :
- **Unicité** : MERGE idempotent → on peut relancer sans créer de doublons
- **Performance** : index sur les propriétés de filtrage fréquentes
- **Recherche fulltext** : index sur preferredLabel et description

In [ ]:
# Afficher les contraintes du schéma
schema_path = SRC / 'schema.cypher'
cypher = schema_path.read_text()

# Compter les statements
constraints = [s.strip() for s in cypher.split(';')
               if 'CREATE CONSTRAINT' in s]
indexes_perf = [s.strip() for s in cypher.split(';')
                if 'CREATE INDEX' in s and 'FULLTEXT' not in s]
indexes_ft   = [s.strip() for s in cypher.split(';')
                if 'FULLTEXT' in s]

print(f'Contraintes d unicite : {len(constraints)}')
print(f'Index de performance  : {len(indexes_perf)}')
print(f'Index fulltext        : {len(indexes_ft)}')
print()
print('Contraintes :')
for c in constraints[:5]:
    name_line = [l for l in c.split('\n') if 'CONSTRAINT' in l]
    if name_line: print(f'  {name_line[0].strip()[:80]}')

if NEO4J_AVAILABLE:
    from load_neo4j import create_schema, get_driver
    drv = get_driver()
    create_schema(drv)
    print('Schema cree')
    drv.close()

## 3. Chargement ESCO — Compétences, Métiers, ISCO, Relations

**13 939 compétences** + **3 039 métiers** + **126 051 relations :NECESSITE**

La logique de chargement :
1. `MERGE` sur `conceptUri` (idempotent)
2. `SET` toutes les propriétés (mise à jour si nœud existant)
3. Enrichissement des compétences avec les collections (isDigital, isGreen, ...)
4. Enrichissement des métiers avec les codes MEPC depuis la table de mapping

In [ ]:
# Simulation du chargement ESCO (sans Neo4j)
import csv

skills_data = []
with open(ESCO_DIR / 'skills_fr.csv') as f:
    for row in csv.DictReader(f):
        skills_data.append(row)

occ_data = []
with open(ESCO_DIR / 'occupations_fr.csv') as f:
    for row in csv.DictReader(f):
        occ_data.append(row)

rel_data = []
with open(ESCO_DIR / 'occupationSkillRelations_fr.csv') as f:
    for row in csv.DictReader(f):
        rel_data.append(row)

print(f'Competences ESCO   : {len(skills_data):,}')
print(f'Metiers ESCO       : {len(occ_data):,}')
print(f'Relations necessite: {len(rel_data):,}')

# Distribution types de compétences
from collections import Counter
sk_types = Counter(r['skillType'] for r in skills_data)
rel_types = Counter(r['relationType'] for r in rel_data)
print(f'\nTypes competences   : {dict(sk_types)}')
print(f'Types relations     : {dict(rel_types)}')

# Nombre moyen de compétences par métier
from collections import defaultdict
skills_per_occ = defaultdict(int)
for r in rel_data:
    skills_per_occ[r['occupationUri']] += 1
counts = list(skills_per_occ.values())
print(f'\nCompetences/metier : moy={np.mean(counts):.1f}'
      f'  med={np.median(counts):.0f}  max={max(counts)}')

if NEO4J_AVAILABLE:
    from load_neo4j import load_esco_skills, load_esco_occupations, load_esco_isco, load_esco_skill_groups, load_esco_relations, get_driver
    drv = get_driver()
    load_esco_skills(drv)
    load_esco_occupations(drv)
    load_esco_isco(drv)
    load_esco_skill_groups(drv)
    load_esco_relations(drv)
    drv.close()
    print('ESCO charge dans Neo4j')

In [ ]:
# Visualisation distribution compétences ESCO
NAVY, TEAL, ORANGE, GREEN = '#1E2761','#028090','#E67E22','#27AE60'

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Structure ESCO v1.2 — Avant chargement Neo4j', fontsize=13, fontweight='bold', color=NAVY)

# Types de skills
axes[0].pie([sk_types.get('skill/competence',0), sk_types.get('knowledge',0)],
            labels=['Compétences\n(skills)', 'Connaissances\n(knowledge)'],
            colors=[TEAL, NAVY], autopct='%1.0f%%',
            wedgeprops=dict(edgecolor='white', lw=2))
axes[0].set_title('Types de skills ESCO', fontweight='bold')

# Relations essential vs optional
axes[1].bar(['Essential', 'Optional'],
            [rel_types.get('essential',0), rel_types.get('optional',0)],
            color=[NAVY, TEAL], edgecolor='white', width=0.5)
axes[1].set_title('Relations :NECESSITE par type', fontweight='bold')
axes[1].set_ylabel('Nombre')
for i, v in enumerate([rel_types.get('essential',0), rel_types.get('optional',0)]):
    axes[1].text(i, v+500, f'{v:,}', ha='center', fontsize=10, fontweight='bold')

# Distribution skills/métier
axes[2].hist(counts, bins=30, color=ORANGE, edgecolor='white', rwidth=0.85)
axes[2].axvline(np.mean(counts), color=NAVY, lw=2, label=f'Moy={np.mean(counts):.0f}')
axes[2].set_title('Distribution skills/métier', fontweight='bold')
axes[2].set_xlabel('Nombre de compétences par métier')
axes[2].set_ylabel('Nombre de métiers')
axes[2].legend()

plt.tight_layout()
plt.savefig('fig_esco_structure.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Chargement MEPC — 3 niveaux + alignement ISCO-08

La MEPC 2013 (INS Cameroun) contextualise les métiers dans la réalité camerounaise.
L'alignement `GroupeBaseMEPC -[:ALIGNE_AVEC]-> GroupeISCO` est le **pont clé** qui
relie MEPC et ESCO via les codes ISCO-08 communs.

In [ ]:
# Visualiser la structure MEPC
mepc_g = pd.read_parquet(PROC / 'mepc_grands_groupes.parquet')
mepc_s = pd.read_parquet(PROC / 'mepc_sous_groupes.parquet')
mepc_b = pd.read_parquet(PROC / 'mepc_groupes_base.parquet')
mapping = pd.read_parquet(PROC / 'mapping_isco_mepc_esco.parquet')

print('=== MEPC 2013 ===')
print(f'  Grands Groupes    : {len(mepc_g)}')
print(f'  Sous-Groupes      : {len(mepc_s)}')
print(f'  Groupes de Base   : {len(mepc_b)}')
print(f'  Total nœuds MEPC  : {len(mepc_g)+len(mepc_s)+len(mepc_b)}')

# Mapping MEPC↔ESCO
n_with_esco = mapping['esco_uri'].notna().sum()
print(f'\n=== MAPPING MEPC->ESCO ===')
print(f'  Correspondances totales  : {len(mapping):,}')
print(f'  Avec URI ESCO mappé      : {n_with_esco:,} ({n_with_esco/len(mapping)*100:.1f}%)')
print(f'  Relation Neo4j produite  : GroupeBaseMEPC -[:ALIGNE_AVEC]-> GroupeISCO')
print(f'  + Métier -[:CORRESPOND_MEPC]-> GroupeBaseMEPC')

# Grand groupe distribution
n_base_per_grand = mepc_b.groupby('code_grand_groupe').size()
print(f'\nGroupes de base par grand groupe :')
for code, n in n_base_per_grand.items():
    label = mepc_g[mepc_g['code']==str(code)]['intitule'].values
    lbl = label[0][:40] if len(label) else str(code)
    print(f'  Groupe {code} — {lbl:<40} : {n} groupes de base')

if NEO4J_AVAILABLE:
    from load_neo4j import load_mepc, get_driver
    drv = get_driver()
    load_mepc(drv)
    drv.close()

## 5. Chargement NCF — 4 niveaux hiérarchiques

In [ ]:
ncf_n = pd.read_parquet(PROC / 'ncf_niveaux.parquet')
ncf_g = pd.read_parquet(PROC / 'ncf_grands_domaines.parquet')
ncf_s = pd.read_parquet(PROC / 'ncf_dom_specialises.parquet')
ncf_d = pd.read_parquet(PROC / 'ncf_dom_detailles.parquet')

print('=== NCF 2017 ===')
for name, df in [('NiveauFormationNCF',9), ('GrandDomaineNCF',10),
                  ('DomaineSpecialiseNCF',31), ('DomaineDétailléNCF',201)]:
    print(f'  {name:<25} : {df} nœuds attendus')

print()
print('Niveaux NCF (correspondance diplômes camerounais) :')
for _, r in ncf_n.iterrows():
    print(f'  NCF-{r["code"]} — {r["intitule"]}')

if NEO4J_AVAILABLE:
    from load_neo4j import load_ncf, get_driver
    drv = get_driver()
    load_ncf(drv)
    drv.close()

## 6. Chargement Offres d'emploi

7 861 offres → nœuds :OffreEmploi + nœuds contextuels (Secteur, Employeur, Localisation)

In [ ]:
df_o = pd.read_parquet(PROC / 'offres_normalized.parquet')
print(f'Offres a charger : {len(df_o):,}')
print(f'Colonnes         : {list(df_o.columns)}')
print()
print('Nœuds contextuels à créer :')
print(f'  Secteurs    : {df_o["secteur_principal"].nunique()}')
print(f'  Employeurs  : {df_o["employeur"].nunique()}')
print(f'  Villes      : {df_o["ville_principale"].nunique()}')

print('\nRelations offre créées :')
print('  OffreEmploi -[:DANS_SECTEUR]-> Secteur')
print('  OffreEmploi -[:PUBLIEE_PAR]-> Employeur')
print('  OffreEmploi -[:LOCALISEE_A]-> Localisation')
print('  OffreEmploi -[:REQUIERT_NIVEAU_NCF]-> NiveauFormationNCF')
print(f'  (+ :REQUIERT -> Compétence via LLM — module 05)')

# Exemple de nœud construit
ex = df_o.iloc[0]
print('\nExemple nœud :OffreEmploi :')
for k in ['offre_id','titre_poste','employeur','ville_principale',
           'secteur_principal','type_contrat_norm','ncf_niveau_code']:
    print(f'  {k:<25} = {ex.get(k)}')

if NEO4J_AVAILABLE:
    from load_neo4j import load_offres, get_driver
    drv = get_driver()
    load_offres(drv)
    drv.close()

## 7. Chargement Candidats

In [ ]:
df_c = pd.read_parquet(PROC / 'candidats_normalized.parquet')
print(f'Candidats a charger : {len(df_c):,}')
print()
# Exemple nœud Candidat
ex = df_c.iloc[0]
print('Exemple nœud :Candidat :')
for k in ['candidat_id','metier_vise','secteur_metier','ncf_niveau_final',
           'filiere_specialite','mobilite_geo_bool','text_to_embed']:
    v = ex.get(k)
    if isinstance(v,str) and len(v)>60: v = v[:60]+'...'
    print(f'  {k:<25} = {v}')

print('\nRelations candidat créées :')
print('  Candidat -[:A_NIVEAU]-> NiveauFormationNCF')
print('  Candidat -[:A_FORMATION]-> DomaineDétailléNCF  (si filière matchée)')
print('  (+ :POSSEDE -> Compétence via déclaration — module 05)')

if NEO4J_AVAILABLE:
    from load_neo4j import load_candidats, get_driver
    drv = get_driver()
    load_candidats(drv)
    drv.close()

## 8. Validation et métriques du graphe

Résultats attendus après chargement complet.

In [ ]:
# Résultats de validation attendus
EXPECTED = {
    'Nœuds': {
        'Candidat': 1105,
        'OffreEmploi': 7861,
        'Compétence': 13939,
        'Métier': 3039,
        'GroupeISCO': 614,
        'GroupeCompétences': 45,
        'GrandGroupeMEPC': 8,
        'SousGroupeMEPC': 44,
        'GroupeBaseMEPC': 209,
        'NiveauFormationNCF': 9,
        'GrandDomaineNCF': 10,
        'DomaineSpécialiséNCF': 31,
        'DomaineDétailléNCF': 201,
        'Secteur': None,  # variable
        'Employeur': None,
        'Localisation': None,
    },
    'Relations': {
        ':NECESSITE (Métier->Comp)': 126051,
        ':CLASSIFIE_DANS (Métier->ISCO)': 3039,
        ':PLUS_LARGE_QUE (Comp->Comp)': 2488,
        ':ALIGNE_AVEC (MEPC->ISCO)': None,
        ':CORRESPOND_MEPC (Métier->MEPC)': None,
        ':CONTIENT MEPC': 253,
        ':CONTIENT NCF': 241,
        ':A_NIVEAU (Cand->NCF)': None,
        ':DANS_SECTEUR (Offre->Sect)': 7861,
        ':PUBLIEE_PAR (Offre->Emp)': 7861,
    }
}

print('=== RÉSULTATS ATTENDUS APRÈS CHARGEMENT COMPLET ===')
total_noeuds = sum(v for v in EXPECTED['Nœuds'].values() if v)
total_rels   = sum(v for v in EXPECTED['Relations'].values() if v)
print(f'\nNœuds (16 types) : ~{total_noeuds:,}')
print(f'Relations attendues (base) : ~{total_rels:,}')

print('\nDétail nœuds :')
for label, n in EXPECTED['Nœuds'].items():
    val = f'{n:,}' if n else 'variable'
    print(f'  :{label:<30} {val:>10}')

print('\nDétail relations :')
for rel, n in EXPECTED['Relations'].items():
    val = f'{n:,}' if n else 'variable'
    print(f'  {rel:<40} {val:>10}')

if NEO4J_AVAILABLE:
    from load_neo4j import validate_graph, get_driver
    drv = get_driver()
    stats = validate_graph(drv)
    drv.close()
    print('Validation OK')

## 9. Requêtes de démonstration

Démonstration des requêtes Cypher utilisées dans le module 05 (GraphRAG).

In [ ]:
from queries_cypher import (
    Q_SKILL_GAP_EXACT, Q_OFFRES_COMPATIBLES, Q_COMPETENCES_A_ACQUERIR,
    Q_CHEMIN_FORMATION, Q_TOP_COMPETENCES_OFFRES, Q_CANDIDATS_SIMILAIRES
)

# Afficher les requêtes clés
queries = [
    ('SKILL GAP EXACT', Q_SKILL_GAP_EXACT[:800]),
    ('OFFRES COMPATIBLES (pré-filtre)', Q_OFFRES_COMPATIBLES[:700]),
    ('COMPÉTENCES À ACQUÉRIR (roadmap)', Q_COMPETENCES_A_ACQUERIR[:600]),
]
for name, q in queries:
    print(f'=== {name} ===')
    print(q)
    print()

In [ ]:
# Simulation d'une requête de skill gap (sans Neo4j)
# Données simulées représentatives du graphe réel

candidat_sim = {
    'id': 'PPKOU2501080016340',
    'metier_vise': 'Agent de transit',
    'competences_possedees': [
        'utiliser des logiciels de gestion de transport',
        'communiquer avec les clients',
        'rédiger des documents administratifs',
        'gérer des fichiers et des dossiers',
    ]
}

offre_sim = {
    'id': 'offre-transit-001',
    'titre': 'Assistant Transit et Douane',
    'competences_requises': {
        'essential': [
            'utiliser des logiciels de gestion de transport',
            'appliquer les procedures douanieres',
            'coordonner les operations logistiques',
            'communiquer avec les clients',
        ],
        'optional': [
            'gerer un budget',
            'utiliser des bases de donnees',
        ]
    }
}

# Calcul du skill gap
cand_set = set(candidat_sim['competences_possedees'])
req_ess  = set(offre_sim['competences_requises']['essential'])
req_opt  = set(offre_sim['competences_requises']['optional'])
req_all  = req_ess | req_opt

acquises  = cand_set & req_all
manquantes = req_all - cand_set
ess_manquantes = req_ess - cand_set
taux = len(acquises) / len(req_all)

print('=== SIMULATION SKILL GAP ===')
print(f'Candidat   : {candidat_sim["id"]} → {candidat_sim["metier_vise"]}')
print(f'Offre      : {offre_sim["titre"]}')
print()
print(f'Taux matching : {taux:.1%} ({len(acquises)}/{len(req_all)} compétences)')
print()
print('Compétences ACQUISES :')
for c in sorted(acquises):    print(f'  ✓ {c}')
print('Compétences MANQUANTES (dont essentielles) :')
for c in sorted(manquantes):
    ess = ' [ESSENTIELLE]' if c in ess_manquantes else ''
    print(f'  ✗ {c}{ess}')

## 10. Visualisation de la structure du graphe

In [ ]:
NAVY, TEAL, ORANGE, GREEN, RED = '#1E2761','#028090','#E67E22','#27AE60','#C0392B'
PURPLE, GRAY = '#534AB7','#95A5A6'

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Structure du Graphe de Connaissances Neo4j\n'
             'Emploi-Compétences Cameroun — 16 nœuds · 22 relations',
             fontsize=13, fontweight='bold', color=NAVY)

# Donut des nœuds par famille
families = {
    'Acteurs\n(Candidat, Offre)': 7861+1105,
    'ESCO\n(Compétence, Métier)': 13939+3039+614+45,
    'MEPC\n(3 niveaux)': 8+44+209,
    'NCF\n(4 niveaux)': 9+10+31+201,
    'Contexte\n(Sect/Emp/Loc)': 400,
}
colors_f = [TEAL, NAVY, RED, GREEN, ORANGE]
wedges, texts, autos = axes[0].pie(
    list(families.values()), labels=list(families.keys()),
    autopct='%1.0f%%', colors=colors_f, startangle=90,
    wedgeprops=dict(edgecolor='white', lw=2), pctdistance=0.8)
for at in autos: at.set_fontsize(8); at.set_fontweight('bold')
axes[0].set_title('Répartition nœuds par famille', fontweight='bold')

# Barres relations
rels = {
    ':NECESSITE': 126051,
    ':PLUS_LARGE_QUE': 2488,
    ':CORRESPOND_MEPC': 19987,
    ':ALIGNE_AVEC': 400,
    ':CLASSIFIE_DANS': 3039,
    ':CONTIENT': 494,
    ':DANS_SECTEUR': 7861,
    ':PUBLIEE_PAR': 7861,
    ':A_NIVEAU': 1100,
    ':PARTIE_DE': 2500,
}
names_r = list(rels.keys())
vals_r  = list(rels.values())
colors_r= [NAVY if v>10000 else (TEAL if v>2000 else ORANGE) for v in vals_r]
y_pos = range(len(names_r))
axes[1].barh(y_pos, vals_r, color=colors_r, edgecolor='white')
axes[1].set_yticks(y_pos); axes[1].set_yticklabels(names_r, fontsize=8)
axes[1].set_xscale('log')
axes[1].set_title('Cardinalité des relations (log)', fontweight='bold')
axes[1].set_xlabel('Nombre (échelle log)')

# Heatmap connexions inter-entités
import numpy as np
entities = ['Candidat','Offre','Compétence','Métier','MEPC','NCF','ISCO']
matrix = np.zeros((7,7))
connections = [
    (0,2,1),(0,5,1),(0,1,1),  # Candidat -> Comp, NCF, Offre
    (1,2,1),(1,3,1),(1,6,1),  # Offre -> Comp, Metier, ISCO
    (3,2,1),(3,4,1),(3,6,1),  # Metier -> Comp, MEPC, ISCO
    (2,2,0.5),                # Comp -> Comp
    (4,6,0.5),(5,3,0.5),      # MEPC -> ISCO, NCF -> Metier
]
for i,j,v in connections:
    matrix[i,j] = v; matrix[j,i] = v*0.5
im = axes[2].imshow(matrix, cmap='YlOrRd', aspect='auto')
axes[2].set_xticks(range(7)); axes[2].set_yticks(range(7))
axes[2].set_xticklabels(entities, rotation=35, fontsize=8)
axes[2].set_yticklabels(entities, fontsize=8)
axes[2].set_title('Matrice de connexion inter-entités', fontweight='bold')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('fig_graphe_structure.png', dpi=140, bbox_inches='tight')
plt.show()

---
## Synthèse du Module 03

| Élément | Valeur |
|---|---|
| **Nœuds chargés** | ~26 500 (16 types) |
| **Relations créées** | ~165 000 (22 types, hors :REQUIERT enrichi par LLM 05) |
| **Garantie d'idempotence** | MERGE sur clé unique → zéro doublon au rechargement |
| **Batch size** | 500 nœuds / 2000 relations par transaction |
| **Temps estimé (chargement complet)** | 5-15 min selon machine |

### Commandes

```bash
# Pipeline complet
python src/03_knowledge_graph/load_neo4j.py

# Étape par étape
python src/03_knowledge_graph/load_neo4j.py --step schema
python src/03_knowledge_graph/load_neo4j.py --step esco
python src/03_knowledge_graph/load_neo4j.py --step mepc
python src/03_knowledge_graph/load_neo4j.py --step ncf
python src/03_knowledge_graph/load_neo4j.py --step offres
python src/03_knowledge_graph/load_neo4j.py --step candidats

# Validation sans écriture
python src/03_knowledge_graph/load_neo4j.py --dry-run
```

### → Prochaine étape : Module 04 — Embeddings pgvector
Encoder toutes les entités (texte_to_embed) avec le ST fine-tuné (Module 02)
et stocker les vecteurs 384d dans PostgreSQL + pgvector avec index HNSW.